# Chapitre 4 — Partie 5 : Optimisation des Hyperparamètres

**Durée estimée : 2h30**

## 🎯 Objectifs d'apprentissage

À la fin de cette partie, vous serez capable de :
1. **Distinguer** paramètres et hyperparamètres
2. **Implémenter** GridSearchCV pour une recherche exhaustive
3. **Utiliser** RandomizedSearchCV pour une exploration efficace
4. **Combiner** Pipeline + CV + Grid Search pour un workflow complet

---

## Lien avec les chapitres précédents

Dans le **Chapitre 2 (Leçon 3)**, nous avons brièvement mentionné les hyperparamètres : ces valeurs qu'on choisit AVANT l'entraînement (comme `max_depth` pour les arbres).

Dans la **Partie 4**, nous avons appris à évaluer un modèle avec la validation croisée.

Maintenant, nous allons **combiner** ces deux concepts pour trouver automatiquement les meilleurs hyperparamètres.

---

## 🌍 Problème Réel : Trop de boutons à régler !

Vous entraînez un Random Forest. Le modèle a de nombreux hyperparamètres :

- `n_estimators` : 10 ? 100 ? 500 ? 1000 ?
- `max_depth` : 5 ? 10 ? 20 ? None ?
- `min_samples_split` : 2 ? 5 ? 10 ?
- `min_samples_leaf` : 1 ? 2 ? 4 ?
- ...

Avec 4 choix pour chaque hyperparamètre, vous avez **4 × 4 × 4 × 4 = 256 combinaisons** possibles !

**Question :** Comment trouver la meilleure combinaison sans passer des heures à tester manuellement ?

*(Réponse attendue : Automatiser la recherche ! Tester systématiquement toutes les combinaisons (ou un échantillon) et garder la meilleure.)*

---

## 5.1 Rappel : Paramètres vs Hyperparamètres

```
┌─────────────────────────────────────────────────────────────────────┐
│           PARAMÈTRES vs HYPERPARAMÈTRES                             │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   PARAMÈTRES (appris pendant l'entraînement)                        │
│   ───────────────────────────────────────────                       │
│   • Coefficients d'une régression linéaire                         │
│   • Poids d'un réseau de neurones                                   │
│   • Seuils de décision d'un arbre                                   │
│                                                                     │
│   → Déterminés par .fit()                                          │
│   → Vous n'avez PAS à les choisir                                  │
│                                                                     │
│   ─────────────────────────────────────────────────────────────────│
│                                                                     │
│   HYPERPARAMÈTRES (choisis AVANT l'entraînement)                    │
│   ────────────────────────────────────────────                      │
│   • n_estimators, max_depth (Random Forest)                        │
│   • C, kernel (SVM)                                                 │
│   • learning_rate, n_neighbors (KNN, etc.)                         │
│                                                                     │
│   → Choisis par VOUS avant .fit()                                  │
│   → Influencent la capacité d'apprentissage du modèle              │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import (train_test_split, cross_val_score, 
                                     GridSearchCV, RandomizedSearchCV)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_classification
import warnings
warnings.filterwarnings('ignore')

# Créer un dataset
np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10,
                           n_redundant=5, random_state=42)

# Split initial (garder un test set final)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("📊 Dataset prêt")
print(f"  Train : {len(X_train)} exemples")
print(f"  Test  : {len(X_test)} exemples")

---

## 5.2 GridSearchCV : La Recherche Exhaustive

### L'idée

GridSearchCV teste **TOUTES** les combinaisons d'hyperparamètres que vous spécifiez, en utilisant la validation croisée pour évaluer chacune.

```
┌─────────────────────────────────────────────────────────────────────┐
│              GRIDSEARCHCV : Le Processus                            │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   Vous définissez une "grille" d'hyperparamètres :                 │
│                                                                     │
│   param_grid = {                                                    │
│       'n_estimators': [50, 100, 200],                              │
│       'max_depth': [5, 10, None]                                    │
│   }                                                                 │
│                                                                     │
│   GridSearchCV teste TOUTES les combinaisons :                      │
│                                                                     │
│   ┌──────────────┬───────────┬──────────────────────────────┐      │
│   │ n_estimators │ max_depth │ Score CV (moyenne ± std)     │      │
│   ├──────────────┼───────────┼──────────────────────────────┤      │
│   │     50       │     5     │       0.82 ± 0.02            │      │
│   │     50       │    10     │       0.84 ± 0.03            │      │
│   │     50       │   None    │       0.83 ± 0.02            │      │
│   │    100       │     5     │       0.83 ± 0.02            │      │
│   │    100       │    10     │       0.86 ± 0.02   ← BEST   │      │
│   │    100       │   None    │       0.85 ± 0.03            │      │
│   │    200       │     5     │       0.83 ± 0.02            │      │
│   │    200       │    10     │       0.86 ± 0.02            │      │
│   │    200       │   None    │       0.85 ± 0.02            │      │
│   └──────────────┴───────────┴──────────────────────────────┘      │
│                                                                     │
│   → 3 × 3 = 9 combinaisons testées                                 │
│   → Avec CV=5 : 9 × 5 = 45 entraînements !                         │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### Implémentation

In [ ]:
# Définir le modèle de base
rf = RandomForestClassifier(random_state=42)

# Définir la grille d'hyperparamètres
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10]
}

# Calculer le nombre de combinaisons
n_combinaisons = 3 * 4 * 3
print(f"📊 Grille de recherche")
print(f"  Combinaisons à tester : {n_combinaisons}")
print(f"  Avec CV=5 : {n_combinaisons * 5} entraînements")

In [ ]:
# Créer le GridSearchCV
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,                    # 5-Fold CV
    scoring='accuracy',      # Métrique à optimiser
    n_jobs=-1,               # Utiliser tous les CPU
    verbose=1                # Afficher la progression
)

# Lancer la recherche
print("\n🔍 Lancement de GridSearchCV...")
grid_search.fit(X_train, y_train)

In [ ]:
# Afficher les meilleurs résultats
print("\n📊 Résultats de GridSearchCV")
print("=" * 60)

print(f"\n🏆 Meilleurs hyperparamètres :")
for param, value in grid_search.best_params_.items():
    print(f"   {param}: {value}")

print(f"\n📈 Meilleur score CV : {grid_search.best_score_:.4f}")
print(f"   (Moyenne sur les 5 folds)")

In [ ]:
# Évaluer sur le test set final
best_model = grid_search.best_estimator_
test_score = best_model.score(X_test, y_test)

print(f"\n🎯 Score sur le TEST SET (jamais vu) : {test_score:.4f}")

### Explorer tous les résultats

In [ ]:
# Convertir les résultats en DataFrame
resultats = pd.DataFrame(grid_search.cv_results_)

# Afficher les colonnes pertinentes
colonnes = ['param_n_estimators', 'param_max_depth', 'param_min_samples_split',
            'mean_test_score', 'std_test_score', 'rank_test_score']

print("📊 Top 10 des combinaisons")
print("=" * 70)
resultats[colonnes].sort_values('rank_test_score').head(10)

<details>
<summary>🤔 Question Socratique : Pourquoi utilise-t-on la CV DANS le GridSearch et pas juste un validation set ?</summary>

### 🔑 Réponse

Avec un validation set unique, le choix des "meilleurs" hyperparamètres dépendrait de quels exemples sont tombés dans le validation set. C'est le même problème de variabilité qu'on a vu avec le train/test split.

En utilisant la **CV dans le GridSearch** :

1. Chaque combinaison est évaluée K fois (sur K folds différents)
2. On obtient une moyenne ET un écart-type
3. Le choix est plus robuste et moins sujet au hasard

**Bonus :** On utilise 100% des données d'entraînement pour la validation (chaque exemple est dans un fold de test exactement une fois).

</details>

---

## 5.3 RandomizedSearchCV : Quand la Grille est Trop Grande

### Le problème de GridSearchCV

Le nombre de combinaisons explose rapidement :

| Hyperparamètres | Valeurs par param | Combinaisons | Avec CV=5 |
|-----------------|-------------------|--------------|----------|
| 3 | 4 | 64 | 320 |
| 5 | 5 | 3 125 | 15 625 |
| 7 | 5 | 78 125 | 390 625 |

**Solution :** Au lieu de tester TOUTES les combinaisons, en tester un **échantillon aléatoire**.

### Implémentation avec des distributions

In [ ]:
from scipy.stats import randint, uniform

# Définir les distributions (pas des listes fixes !)
param_distributions = {
    'n_estimators': randint(50, 500),           # Entier entre 50 et 500
    'max_depth': randint(3, 30),                # Entier entre 3 et 30
    'min_samples_split': randint(2, 20),        # Entier entre 2 et 20
    'min_samples_leaf': randint(1, 10),         # Entier entre 1 et 10
    'max_features': ['sqrt', 'log2', None]      # Catégoriel
}

print("📊 Espaces de recherche")
print("=" * 50)
print(f"  n_estimators     : 50 - 500 (uniforme)")
print(f"  max_depth        : 3 - 30 (uniforme)")
print(f"  min_samples_split: 2 - 20 (uniforme)")
print(f"  min_samples_leaf : 1 - 10 (uniforme)")
print(f"  max_features     : ['sqrt', 'log2', None]")

In [ ]:
# Créer le RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_distributions,
    n_iter=50,               # Nombre de combinaisons à tester
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("\n🔍 Lancement de RandomizedSearchCV (50 itérations)...")
random_search.fit(X_train, y_train)

In [ ]:
# Afficher les résultats
print("\n📊 Résultats de RandomizedSearchCV")
print("=" * 60)

print(f"\n🏆 Meilleurs hyperparamètres :")
for param, value in random_search.best_params_.items():
    print(f"   {param}: {value}")

print(f"\n📈 Meilleur score CV : {random_search.best_score_:.4f}")

# Test set
test_score_random = random_search.best_estimator_.score(X_test, y_test)
print(f"\n🎯 Score sur le TEST SET : {test_score_random:.4f}")

### Comparaison : GridSearch vs RandomizedSearch

In [ ]:
print("""
┌─────────────────────────────────────────────────────────────────────┐
│           GRIDSEARCH vs RANDOMIZEDSEARCH                            │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   GRIDSEARCHCV                                                      │
│   ─────────────                                                     │
│   ✅ Teste TOUTES les combinaisons                                  │
│   ✅ Garantit de trouver l'optimum dans la grille                  │
│   ❌ Très lent si beaucoup d'hyperparamètres                       │
│   ❌ Ne teste que les valeurs spécifiées                           │
│                                                                     │
│   → Idéal pour : peu d'hyperparamètres, valeurs discrètes          │
│                                                                     │
│   ─────────────────────────────────────────────────────────────────│
│                                                                     │
│   RANDOMIZEDSEARCHCV                                                │
│   ───────────────────                                               │
│   ✅ Contrôle du temps (n_iter fixe)                               │
│   ✅ Explore un espace continu                                      │
│   ✅ Souvent aussi bon que GridSearch en moins de temps            │
│   ❌ Peut manquer l'optimum (mais peu probable)                     │
│                                                                     │
│   → Idéal pour : beaucoup d'hyperparamètres, exploration initiale  │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
""")

---

## 5.4 Pipeline + GridSearchCV : Le Workflow Complet

### Le problème du data leakage

Si vous faites le preprocessing **avant** GridSearchCV, vous avez un data leakage ! Le scaler ou encoder aura "vu" les données de validation.

**Solution :** Mettre le preprocessing **dans** un Pipeline, puis faire GridSearchCV sur le pipeline.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# Créer un pipeline complet
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42))
])

print("📊 Pipeline créé")
print(pipeline)

In [ ]:
# Définir la grille pour le pipeline
# Note : les noms des paramètres utilisent le format 'step__param'
param_grid_pipeline = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [5, 10, None],
    'classifier__min_samples_split': [2, 5]
}

print("📊 Grille pour le pipeline")
print("  (Notez le préfixe 'classifier__' pour les params du RF)")
for k, v in param_grid_pipeline.items():
    print(f"  {k}: {v}")

In [ ]:
# GridSearchCV sur le pipeline complet
grid_pipeline = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid_pipeline,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

print("\n🔍 Lancement de GridSearchCV sur le Pipeline...")
grid_pipeline.fit(X_train, y_train)

In [ ]:
# Résultats
print("\n📊 Résultats")
print("=" * 60)
print(f"\n🏆 Meilleurs hyperparamètres :")
for param, value in grid_pipeline.best_params_.items():
    print(f"   {param}: {value}")

print(f"\n📈 Meilleur score CV : {grid_pipeline.best_score_:.4f}")
print(f"🎯 Score TEST : {grid_pipeline.best_estimator_.score(X_test, y_test):.4f}")

---

```
┌─────────────────────────────────────────────────────────────────────┐
│           WORKFLOW COMPLET : Pipeline + GridSearchCV                │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   1. SÉPARER les données (train / test final)                       │
│      X_train, X_test = train_test_split(...)                        │
│                                                                     │
│   2. CRÉER un Pipeline avec preprocessing + modèle                  │
│      pipeline = Pipeline([                                          │
│          ('preprocesseur', ...),                                    │
│          ('classifier', RandomForest())                             │
│      ])                                                             │
│                                                                     │
│   3. DÉFINIR la grille d'hyperparamètres                           │
│      param_grid = {'classifier__param': [val1, val2]}              │
│                                                                     │
│   4. LANCER GridSearchCV sur le pipeline                           │
│      grid = GridSearchCV(pipeline, param_grid, cv=5)               │
│      grid.fit(X_train, y_train)                                    │
│                                                                     │
│   5. ÉVALUER sur le test set final                                 │
│      grid.best_estimator_.score(X_test, y_test)                    │
│                                                                     │
│   → Le preprocessing est fait DANS chaque fold = pas de leakage !  │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

---

## 5.5 Bonnes Pratiques

```
┌─────────────────────────────────────────────────────────────────────┐
│           BONNES PRATIQUES - OPTIMISATION HYPERPARAMÈTRES           │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   ✅ À FAIRE                                                        │
│   ────────────                                                      │
│   • Toujours garder un TEST SET final (jamais vu par GridSearch)   │
│   • Utiliser un Pipeline pour éviter le data leakage              │
│   • Commencer par RandomizedSearch (exploration), puis GridSearch  │
│   • Fixer random_state pour la reproductibilité                    │
│   • Utiliser n_jobs=-1 pour paralléliser                           │
│                                                                     │
│   ❌ À ÉVITER                                                       │
│   ─────────────                                                     │
│   • Faire le preprocessing AVANT GridSearchCV                      │
│   • Tester sur le même set utilisé pour la sélection d'hyper      │
│   • Créer des grilles trop grandes (explosion combinatoire)        │
│   • Ignorer l'écart-type des scores CV                             │
│                                                                     │
│   💡 STRATÉGIE RECOMMANDÉE                                          │
│   ─────────────────────────                                         │
│   1. RandomizedSearch (n_iter=50-100) pour explorer large          │
│   2. Analyser les résultats, identifier les zones prometteuses     │
│   3. GridSearch fin autour des meilleures valeurs trouvées         │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

---

## 🧪 Exercice Final : Optimiser un Modèle de Bout en Bout

Vous devez construire le meilleur modèle possible pour prédire la survie de passagers (inspiré du Titanic).

In [ ]:
# Créer un dataset inspiré du Titanic
np.random.seed(42)
n = 800

data = pd.DataFrame({
    'classe': np.random.choice([1, 2, 3], n, p=[0.2, 0.3, 0.5]),
    'age': np.random.uniform(1, 80, n),
    'sexe': np.random.choice([0, 1], n),
    'famille': np.random.randint(0, 6, n),
    'tarif': np.random.uniform(5, 500, n)
})

# Simuler la survie
survie_proba = 0.3 * (data['sexe'] == 1) + 0.2 * (data['age'] < 15) + 0.2 * (data['classe'] == 1) + 0.1
data['survie'] = (np.random.random(n) < survie_proba).astype(int)

# Séparer features et target
X_ex = data.drop('survie', axis=1)
y_ex = data['survie']

# Split
X_train_ex, X_test_ex, y_train_ex, y_test_ex = train_test_split(
    X_ex, y_ex, test_size=0.2, random_state=42, stratify=y_ex
)

print("📊 Dataset 'Survie'")
print("=" * 50)
print(f"  Train : {len(X_train_ex)} exemples")
print(f"  Test  : {len(X_test_ex)} exemples")
print(f"  Taux de survie : {y_ex.mean():.1%}")

### Votre mission :

1. Créez un Pipeline avec StandardScaler + RandomForestClassifier
2. Définissez une grille d'hyperparamètres raisonnable
3. Lancez GridSearchCV avec cv=5 et scoring='f1'
4. Affichez les meilleurs hyperparamètres et le score final
5. Comparez le meilleur modèle à une Logistic Regression de base

In [ ]:
# 🎯 À VOUS DE JOUER !

# Étape 1 : Pipeline
# ...

# Étape 2 : Grille
# ...

# Étape 3 : GridSearchCV
# ...

# Étape 4 : Résultats
# ...

# Étape 5 : Comparaison
# ...

### 🔑 Solution

In [ ]:
# Solution - Étape 1 : Pipeline
pipeline_ex = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=42))
])

print("✅ Pipeline créé")

In [ ]:
# Solution - Étape 2 : Grille
param_grid_ex = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [3, 5, 10, None],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4]
}

n_comb = 3 * 4 * 3 * 3
print(f"✅ Grille définie : {n_comb} combinaisons")

In [ ]:
# Solution - Étape 3 : GridSearchCV
grid_ex = GridSearchCV(
    estimator=pipeline_ex,
    param_grid=param_grid_ex,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

print("\n🔍 Lancement de GridSearchCV...")
grid_ex.fit(X_train_ex, y_train_ex)

In [ ]:
# Solution - Étape 4 : Résultats
from sklearn.metrics import f1_score, classification_report

print("\n📊 Résultats GridSearchCV")
print("=" * 60)

print(f"\n🏆 Meilleurs hyperparamètres :")
for param, value in grid_ex.best_params_.items():
    print(f"   {param}: {value}")

print(f"\n📈 Meilleur score CV (F1) : {grid_ex.best_score_:.4f}")

# Test final
y_pred_ex = grid_ex.best_estimator_.predict(X_test_ex)
f1_test = f1_score(y_test_ex, y_pred_ex)
print(f"🎯 Score TEST (F1) : {f1_test:.4f}")

In [ ]:
# Solution - Étape 5 : Comparaison
print("\n📊 Comparaison avec Logistic Regression (baseline)")
print("=" * 60)

# Baseline
lr_baseline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(random_state=42, max_iter=1000))
])
lr_baseline.fit(X_train_ex, y_train_ex)
y_pred_lr = lr_baseline.predict(X_test_ex)
f1_lr = f1_score(y_test_ex, y_pred_lr)

print(f"\n  Logistic Regression (baseline) : F1 = {f1_lr:.4f}")
print(f"  Random Forest (optimisé)       : F1 = {f1_test:.4f}")
print(f"\n  Amélioration : +{(f1_test - f1_lr)*100:.1f} points de F1")

if f1_test > f1_lr:
    print("\n  🏆 Le Random Forest optimisé surpasse la baseline !")
else:
    print("\n  ⚠️ La baseline est compétitive — considérer sa simplicité")

---

## 🧠 Réflexion Métacognitive

Avant de conclure ce chapitre :

1. **Pourquoi** le test set final ne doit JAMAIS être utilisé pendant le GridSearch ?

2. **Quand** préféreriez-vous RandomizedSearch à GridSearch ?

3. **Comment** éviter le data leakage lors de l'optimisation des hyperparamètres ?

---

## 📝 Résumé du Chapitre 4

| Partie | Concept clé | Ce que vous savez faire |
|--------|-------------|------------------------|
| 4.1 | Métriques Régression | Calculer MAE, RMSE, R² |
| 4.2 | Métriques Classification | Precision, Recall, F1, ROC-AUC |
| 4.3 | Biais-Variance | Diagnostiquer underfitting/overfitting |
| 4.4 | Validation Croisée | Évaluer de manière robuste avec CV |
| 4.5 | Optimisation Hyperparamètres | GridSearchCV, RandomizedSearchCV |

**Code essentiel de ce chapitre :**
```python
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline

# Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier())
])

# GridSearchCV
param_grid = {'model__n_estimators': [50, 100], 'model__max_depth': [5, 10]}
grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='f1')
grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)
print(grid.best_estimator_.score(X_test, y_test))
```

---

## 🎉 Félicitations !

Vous avez terminé le **Chapitre 4 : Évaluation et Optimisation**.

Vous maîtrisez maintenant :
- Les métriques de régression et classification
- Le diagnostic underfitting/overfitting
- La validation croisée
- L'optimisation des hyperparamètres

---

## ➡️ Prochain chapitre

Dans le **Chapitre 5 : Réseaux de Neurones (Conceptuel)**, nous allons découvrir les bases des réseaux de neurones et comprendre comment ils apprennent.

**Question de transition :** Les algorithmes vus jusqu'ici (régression, arbres, etc.) ont des limites. Comment un ordinateur pourrait-il "voir" une image ou comprendre du texte ?